# 基于你自己数据的 Azure 聊天补全模型（预览版）
本示例展示如何将 Azure OpenAI 服务模型与你自己的数据结合使用。该功能目前处于预览阶段。 

Azure OpenAI on your data 让你无需训练或微调模型，就能在自己的数据上运行受支持的聊天模型，例如 GPT-3.5-Turbo 和 GPT-4。在你的数据上运行模型，可以让你基于数据进行对话，并以更高的准确性和速度分析数据。Azure OpenAI on your data 的一个关键优势是能够定制对话式 AI 的内容。由于模型可以访问并引用特定来源来支撑其回复，答案不仅基于其预训练知识，还基于指定数据源中可用的最新信息。这些依据数据（grounding data）也有助于模型避免基于过时或错误的信息生成回复。

结合 Azure AI Search（前身为 Azure Cognitive Search）的 Azure OpenAI on your own data，提供了一个可定制的预构建知识检索解决方案，可在此基础上构建对话式 AI 应用。如需了解其他知识检索和语义搜索方法，请查看 [向量数据库](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases) 的 cookbook 示例。

## 工作原理

[Azure OpenAI on your own data](https://learn.microsoft.com/azure/ai-services/openai/concepts/use-your-data) 将模型与你的数据连接起来，使其能够以增强模型输出的方式检索和利用数据。结合 Azure AI Search，系统会根据用户输入和提供的对话历史，从指定数据源中检索数据。然后对这些数据进行增强，并作为提示重新提交给模型，为模型提供可用于生成回复的上下文信息。

更多信息请参阅 [Data, privacy, and security for Azure OpenAI Service](https://learn.microsoft.com/legal/cognitive-services/openai/data-privacy?context=%2Fazure%2Fai-services%2Fopenai%2Fcontext%2Fcontext)。

## 前提条件
开始之前，我们先介绍几个前提条件。 

为了正确访问 Azure OpenAI 服务，我们需要在 [Azure Portal](https://portal.azure.com) 中创建相应的资源（你可以在 [Microsoft Docs](https://learn.microsoft.com/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal) 中查看详细操作指南）

要将你自己的数据与 Azure OpenAI 模型结合使用，你需要：

1. Azure OpenAI 访问权限，以及一个已部署聊天模型（例如 GPT-3 或 GPT-4）的资源
2. Azure AI Search（前身为 Azure Cognitive Search）资源
3. Azure Blob Storage 资源
4. 用作数据的文档（参见[数据源选项](https://learn.microsoft.com/azure/ai-services/openai/concepts/use-your-data#data-source-options)）


有关如何将文档上传到 Blob 存储并使用 Azure AI Studio 创建索引的完整演练，请参阅此[快速入门](https://learn.microsoft.com/azure/ai-services/openai/use-your-data-quickstart?pivots=programming-language-studio&tabs=command-line)。

## 设置

首先，安装必要的依赖。

In [ ]:
! pip install "openai>=1.0.0,<2.0.0"
! pip install python-dotenv

在本示例中，我们将使用 `dotenv` 加载环境变量。要连接 Azure OpenAI 和搜索索引，应以 `KEY=VALUE` 格式将以下变量添加到 `.env` 文件中：

* `AZURE_OPENAI_ENDPOINT` - Azure OpenAI 终结点。可在 Azure Portal 中你的 Azure OpenAI 资源的 "Keys and Endpoints" 下找到。
* `AZURE_OPENAI_API_KEY` - Azure OpenAI API 密钥。可在 Azure Portal 中你的 Azure OpenAI 资源的 "Keys and Endpoints" 下找到。如果使用 Azure Active Directory 身份验证则可省略（参见下文 `Authentication using Microsoft Active Directory`）
* `SEARCH_ENDPOINT` - AI Search 终结点。该 URL 可在 Azure Portal 中你的 Search 资源的 "Overview" 页面上找到。
* `SEARCH_KEY` - AI Search API 密钥。可在 Azure Portal 中你的 Search 资源的 "Keys" 下找到。
* `SEARCH_INDEX_NAME` - 你使用自己的数据创建的索引名称。

In [ ]:
import os
import openai
import dotenv

dotenv.load_dotenv()

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure Active Directory 令牌凭据。

In [2]:
use_azure_active_directory = False  # Set this flag to True if you are using Azure Active Directory

#### 使用 API 密钥进行身份验证

要将 OpenAI SDK 设置为使用 *Azure API 密钥*，需要将 `api_key` 设置为与你的终结点关联的密钥（你可以在 [Azure Portal](https://portal.azure.com) 中 *"Resource Management"* 下的 *"Keys and Endpoints"* 里找到该密钥）。你也会在这里找到资源的终结点。

In [3]:
if not use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    # set the deployment name for the model we want to use
    deployment = "<deployment-id-of-the-model-to-use>"

    client = openai.AzureOpenAI(
        base_url=f"{endpoint}/openai/deployments/{deployment}/extensions",
        api_key=api_key,
        api_version="2023-09-01-preview"
    )

#### 使用 Azure Active Directory 进行身份验证
现在来看如何通过 Azure Active Directory 进行身份验证。首先安装 `azure-identity` 库。该库会提供身份验证所需的令牌凭据，并通过 `get_bearer_token_provider` 辅助函数帮助我们构建令牌凭据提供程序。建议使用 `get_bearer_token_provider`，而不是向 `AzureOpenAI` 提供静态令牌，因为该 API 会自动为你缓存和刷新令牌。 

有关如何为 Azure OpenAI 设置 Azure Active Directory 身份验证的更多信息，请参阅[文档](https://learn.microsoft.com/azure/ai-services/openai/how-to/managed-identity)。

In [ ]:
! pip install "azure-identity>=1.15.0"

In [5]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

if use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    # set the deployment name for the model we want to use
    deployment = "<deployment-id-of-the-model-to-use>"

    client = openai.AzureOpenAI(
        base_url=f"{endpoint}/openai/deployments/{deployment}/extensions",
        azure_ad_token_provider=get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"),
        api_version="2023-09-01-preview"
    )

> 注意：如果未提供以下参数，AzureOpenAI 会从对应的环境变量中推断它们：

- `api_key` 来自 `AZURE_OPENAI_API_KEY`
- `azure_ad_token` 来自 `AZURE_OPENAI_AD_TOKEN`
- `api_version` 来自 `OPENAI_API_VERSION`
- `azure_endpoint` 来自 `AZURE_OPENAI_ENDPOINT`


## 基于你自己数据的聊天补全模型

### 设置上下文

在本示例中，我们希望模型基于 Azure AI 服务的文档数据进行回复。按照前面分享的[快速入门](https://learn.microsoft.com/azure/ai-services/openai/use-your-data-quickstart?tabs=command-line&pivots=programming-language-studio)，我们已将 [Azure AI services and machine learning](https://learn.microsoft.com/azure/ai-services/cognitive-services-and-machine-learning) 文档页面对应的 [markdown](https://github.com/MicrosoftDocs/azure-docs/blob/main/articles/ai-services/cognitive-services-and-machine-learning.md) 文件添加到搜索索引中。现在，模型已经可以回答有关 Azure AI 服务和机器学习的问题了。

### 代码

现在，我们可以将 Azure on your own data 与 Chat Completions 结合使用。在 `dataSources` 中提供搜索终结点、密钥和索引名称后，向模型提出的任何问题都将以我们自己的数据为依据。响应中还会提供一个额外属性 `context`，用于展示模型回答问题时所引用的数据。

In [ ]:
completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "What are the differences between Azure Machine Learning and Azure AI services?"}],
    model=deployment,
    extra_body={
        "dataSources": [
            {
                "type": "AzureCognitiveSearch",
                "parameters": {
                    "endpoint": os.environ["SEARCH_ENDPOINT"],
                    "key": os.environ["SEARCH_KEY"],
                    "indexName": os.environ["SEARCH_INDEX_NAME"],
                }
            }
        ]
    }
)
print(f"{completion.choices[0].message.role}: {completion.choices[0].message.content}")

# `context` is in the model_extra for Azure
print(f"\nContext: {completion.choices[0].message.model_extra['context']['messages'][0]['content']}")

如果你希望以流式方式获取模型的响应，可以传入关键字参数 `stream=True`：

In [ ]:
response = client.chat.completions.create(
    messages=[{"role": "user", "content": "What are the differences between Azure Machine Learning and Azure AI services?"}],
    model=deployment,
    extra_body={
        "dataSources": [
            {
                "type": "AzureCognitiveSearch",
                "parameters": {
                    "endpoint": os.environ["SEARCH_ENDPOINT"],
                    "key": os.environ["SEARCH_KEY"],
                    "indexName": os.environ["SEARCH_INDEX_NAME"],
                }
            }
        ]
    },
    stream=True,
)

for chunk in response:
    delta = chunk.choices[0].delta

    if delta.role:
        print("\n"+ delta.role + ": ", end="", flush=True)
    if delta.content:
        print(delta.content, end="", flush=True)
    if delta.model_extra.get("context"):
        print(f"Context: {delta.model_extra['context']}", end="", flush=True)